In [0]:
CREATE OR REPLACE TEMP VIEW silver_ads AS

SELECT
    'Amazon' AS platform,
    date,
    campaign_id,
    campaign_name,
    brand,
    product,
    impressions,
    clicks,
    spend,
    orders,
    sales
FROM parquet.`abfss://silver@marketingde2026.dfs.core.windows.net/amazon/amazon_ads.parquet`

UNION ALL

SELECT
    'Flipkart' AS platform,
    date,
    campaign_id,
    campaign_name,
    brand,
    product,
    impressions,
    clicks,
    spend,
    orders,
    sales
FROM parquet.`abfss://silver@marketingde2026.dfs.core.windows.net/flipkart/flipkart_ads.parquet`

UNION ALL

SELECT
    'Blinkit' AS platform,
    date,
    campaign_id,
    campaign_name,
    brand,
    product,
    impressions,
    clicks,
    spend,
    orders,
    sales
FROM parquet.`abfss://silver@marketingde2026.dfs.core.windows.net/blinkit/blinkit_ads.parquet`

UNION ALL

SELECT
    'Zepto' AS platform,
    date,
    campaign_id,
    campaign_name,
    brand,
    product,
    impressions,
    clicks,
    spend,
    orders,
    sales
FROM parquet.`abfss://silver@marketingde2026.dfs.core.windows.net/zepto/zepto_ads.parquet`

UNION ALL

SELECT
    'Swiggy' AS platform,
    date,
    campaign_id,
    campaign_name,
    brand,
    product,
    impressions,
    clicks,
    spend,
    orders,
    sales
FROM parquet.`abfss://silver@marketingde2026.dfs.core.windows.net/swiggy/swiggy_ads.parquet`;

In [0]:
SELECT
    platform,
    COUNT(*) AS record_count
FROM silver_ads
GROUP BY platform
ORDER BY platform;

platform,record_count
Amazon,60
Blinkit,60
Flipkart,60
Swiggy,60
Zepto,60


In [0]:
SELECT
    platform,
    campaign_id,
    campaign_name,
    brand,
    product,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(
        SUM(clicks) / NULLIF(SUM(impressions), 0) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(spend) / NULLIF(SUM(clicks), 0),
        2
    ) AS cpc,

    ROUND(
        SUM(orders) / NULLIF(SUM(clicks), 0) * 100,
        2
    ) AS conversion_rate,

    ROUND(
        SUM(sales) / NULLIF(SUM(spend), 0),
        2
    ) AS roas,

    ROUND(
        SUM(sales) / NULLIF(SUM(orders), 0),
        2
    ) AS aov

FROM silver_ads

GROUP BY
    platform,
    campaign_id,
    campaign_name,
    brand,
    product

ORDER BY roas DESC;

platform,campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,conversion_rate,roas,aov
Blinkit,BLI_PUL_NB,Pulse Notebook - Blinkit,Pulse,Notebook,1.6827596E7,652797.0,2069795.0,8316.0,6430093.91,3.88,3.17,1.27,3.11,773.22
Blinkit,BLI_VER_NB,Verite Notebook - Blinkit,Verite,Notebook,2.3087753E7,838073.0,2678180.0,11411.0,8194306.75,3.63,3.2,1.36,3.06,718.11
Blinkit,BLI_CM_GB,Classmate Geometry Box - Blinkit,Classmate,Geometry Box,2.6083892E7,881190.0,2843145.0,12932.0,8571442.5,3.38,3.23,1.47,3.01,662.81
Blinkit,BLI_CM_PEN,Classmate Pen - Blinkit,Classmate,Pen,2.7929113E7,874261.0,2848770.0,13918.0,8459765.34,3.13,3.26,1.59,2.97,607.83
Blinkit,BLI_CM_NB,Classmate Notebook - Blinkit,Classmate,Notebook,5.6199312E7,1617904.0,5338935.0,28259.0,1.56145088E7,2.88,3.3,1.75,2.92,552.55
Amazon,AMA_PUL_NB,Pulse Notebook - Amazon,Pulse,Notebook,1.0912861E7,423237.0,1342282.5,4302.0,3327733.68,3.88,3.17,1.02,2.48,773.53
Amazon,AMA_VER_NB,Verite Notebook - Amazon,Verite,Notebook,1.4981609E7,543687.0,1737867.5,5911.0,4243917.72,3.63,3.2,1.09,2.44,717.97
Amazon,AMA_CM_GB,Classmate Geometry Box - Amazon,Classmate,Geometry Box,1.692082E7,571476.0,1844370.0,6693.0,4437653.4,3.38,3.23,1.17,2.41,663.03
Amazon,AMA_CM_PEN,Classmate Pen - Amazon,Classmate,Pen,1.812176E7,567087.0,1848420.0,7206.0,4380953.13,3.13,3.26,1.27,2.37,607.96
Amazon,AMA_CM_NB,Classmate Notebook - Amazon,Classmate,Notebook,3.6456944E7,1049227.0,3463410.0,14632.0,8084034.52,2.88,3.3,1.39,2.33,552.49


In [0]:
INSERT OVERWRITE TABLE marketing_gold_campaign_performance
(
    campaign_id,
    campaign_name,
    brand,
    product,
    total_impressions,
    total_clicks,
    total_spend,
    total_orders,
    total_sales,
    ctr,
    cpc,
    roas,
    aov,
    platform
)

SELECT
    campaign_id,
    campaign_name,
    brand,
    product,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(
        SUM(clicks) / NULLIF(SUM(impressions), 0) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(spend) / NULLIF(SUM(clicks), 0),
        2
    ) AS cpc,

    ROUND(
        SUM(sales) / NULLIF(SUM(spend), 0),
        2
    ) AS roas,

    ROUND(
        SUM(sales) / NULLIF(SUM(orders), 0),
        2
    ) AS aov,

    platform

FROM silver_ads

GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product,
    platform;

num_affected_rows,num_inserted_rows
25,25


In [0]:
SELECT
    platform,
    campaign_id,
    campaign_name,
    brand,
    product,
    total_impressions,
    total_clicks,
    total_spend,
    total_orders,
    total_sales,
    ctr,
    cpc,
    roas,
    aov
FROM marketing_gold_campaign_performance
ORDER BY platform, roas DESC;

platform,campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
Amazon,AMA_PUL_NB,Pulse Notebook - Amazon,Pulse,Notebook,10912861,423237.0,1342282.5,4302,3327733.68,3.88,3.17,2.48,773.53
Amazon,AMA_VER_NB,Verite Notebook - Amazon,Verite,Notebook,14981609,543687.0,1737867.5,5911,4243917.72,3.63,3.2,2.44,717.97
Amazon,AMA_CM_GB,Classmate Geometry Box - Amazon,Classmate,Geometry Box,16920820,571476.0,1844370.0,6693,4437653.4,3.38,3.23,2.41,663.03
Amazon,AMA_CM_PEN,Classmate Pen - Amazon,Classmate,Pen,18121760,567087.0,1848420.0,7206,4380953.13,3.13,3.26,2.37,607.96
Amazon,AMA_CM_NB,Classmate Notebook - Amazon,Classmate,Notebook,36456944,1049227.0,3463410.0,14632,8084034.52,2.88,3.3,2.33,552.49
Blinkit,BLI_PUL_NB,Pulse Notebook - Blinkit,Pulse,Notebook,16827596,652797.0,2069795.0,8316,6430093.91,3.88,3.17,3.11,773.22
Blinkit,BLI_VER_NB,Verite Notebook - Blinkit,Verite,Notebook,23087753,838073.0,2678180.0,11411,8194306.75,3.63,3.2,3.06,718.11
Blinkit,BLI_CM_GB,Classmate Geometry Box - Blinkit,Classmate,Geometry Box,26083892,881190.0,2843145.0,12932,8571442.5,3.38,3.23,3.01,662.81
Blinkit,BLI_CM_PEN,Classmate Pen - Blinkit,Classmate,Pen,27929113,874261.0,2848770.0,13918,8459765.34,3.13,3.26,2.97,607.83
Blinkit,BLI_CM_NB,Classmate Notebook - Blinkit,Classmate,Notebook,56199312,1617904.0,5338935.0,28259,1.56145088E7,2.88,3.3,2.92,552.55


In [0]:
SELECT
    platform,

    COUNT(*) AS total_campaigns,

    SUM(total_impressions) AS total_impressions,

    SUM(total_clicks) AS total_clicks,

    ROUND(SUM(total_spend), 2) AS total_spend,

    SUM(total_orders) AS total_orders,

    ROUND(SUM(total_sales), 2) AS total_sales,

    ROUND(
        SUM(total_clicks) / NULLIF(SUM(total_impressions), 0) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(total_spend) / NULLIF(SUM(total_clicks), 0),
        2
    ) AS cpc,

    ROUND(
        SUM(total_sales) / NULLIF(SUM(total_spend), 0),
        2
    ) AS roas,

    ROUND(
        SUM(total_sales) / NULLIF(SUM(total_orders), 0),
        2
    ) AS aov

FROM marketing_gold_campaign_performance

GROUP BY platform

ORDER BY total_sales DESC;

platform,total_campaigns,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
Blinkit,5,150127666,4864225.0,1.5778825E7,74836,4.72701173E7,3.24,3.24,3.0,631.65
Amazon,5,97393994,3154714.0,1.023635E7,38744,2.447429245E7,3.24,3.24,2.39,631.69
Flipkart,5,79912778,2588485.0,8398985.0,29137,1.841275112E7,3.24,3.24,2.19,631.94
Zepto,5,66611948,2157617.0,7000997.5,23128,1.461913105E7,3.24,3.24,2.09,632.1
Swiggy,5,54827915,1775999.0,5762507.5,17963,1.135828535E7,3.24,3.24,1.97,632.32


In [0]:
SELECT
    platform,
    campaign_id,
    campaign_name,
    brand,
    product,
    total_impressions,
    total_clicks,
    total_spend,
    total_orders,
    total_sales,
    ctr,
    cpc,
    roas,
    aov
FROM marketing_gold_campaign_performance
ORDER BY
    platform,
    roas DESC;

platform,campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
Amazon,AMA_PUL_NB,Pulse Notebook - Amazon,Pulse,Notebook,10912861,423237.0,1342282.5,4302,3327733.68,3.88,3.17,2.48,773.53
Amazon,AMA_VER_NB,Verite Notebook - Amazon,Verite,Notebook,14981609,543687.0,1737867.5,5911,4243917.72,3.63,3.2,2.44,717.97
Amazon,AMA_CM_GB,Classmate Geometry Box - Amazon,Classmate,Geometry Box,16920820,571476.0,1844370.0,6693,4437653.4,3.38,3.23,2.41,663.03
Amazon,AMA_CM_PEN,Classmate Pen - Amazon,Classmate,Pen,18121760,567087.0,1848420.0,7206,4380953.13,3.13,3.26,2.37,607.96
Amazon,AMA_CM_NB,Classmate Notebook - Amazon,Classmate,Notebook,36456944,1049227.0,3463410.0,14632,8084034.52,2.88,3.3,2.33,552.49
Blinkit,BLI_PUL_NB,Pulse Notebook - Blinkit,Pulse,Notebook,16827596,652797.0,2069795.0,8316,6430093.91,3.88,3.17,3.11,773.22
Blinkit,BLI_VER_NB,Verite Notebook - Blinkit,Verite,Notebook,23087753,838073.0,2678180.0,11411,8194306.75,3.63,3.2,3.06,718.11
Blinkit,BLI_CM_GB,Classmate Geometry Box - Blinkit,Classmate,Geometry Box,26083892,881190.0,2843145.0,12932,8571442.5,3.38,3.23,3.01,662.81
Blinkit,BLI_CM_PEN,Classmate Pen - Blinkit,Classmate,Pen,27929113,874261.0,2848770.0,13918,8459765.34,3.13,3.26,2.97,607.83
Blinkit,BLI_CM_NB,Classmate Notebook - Blinkit,Classmate,Notebook,56199312,1617904.0,5338935.0,28259,1.56145088E7,2.88,3.3,2.92,552.55


In [0]:
SELECT
    platform,
    COUNT(*) AS total_campaigns,
    SUM(total_impressions) AS total_impressions,
    SUM(total_clicks) AS total_clicks,
    ROUND(SUM(total_spend), 2) AS total_spend,
    SUM(total_orders) AS total_orders,
    ROUND(SUM(total_sales), 2) AS total_sales,

    ROUND(
        SUM(total_clicks) / NULLIF(SUM(total_impressions), 0) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(total_spend) / NULLIF(SUM(total_clicks), 0),
        2
    ) AS cpc,

    ROUND(
        SUM(total_sales) / NULLIF(SUM(total_spend), 0),
        2
    ) AS roas,

    ROUND(
        SUM(total_sales) / NULLIF(SUM(total_orders), 0),
        2
    ) AS aov

FROM marketing_gold_campaign_performance

GROUP BY platform

ORDER BY total_sales DESC;

platform,total_campaigns,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
Blinkit,5,150127666,4864225.0,1.5778825E7,74836,4.72701173E7,3.24,3.24,3.0,631.65
Amazon,5,97393994,3154714.0,1.023635E7,38744,2.447429245E7,3.24,3.24,2.39,631.69
Flipkart,5,79912778,2588485.0,8398985.0,29137,1.841275112E7,3.24,3.24,2.19,631.94
Zepto,5,66611948,2157617.0,7000997.5,23128,1.461913105E7,3.24,3.24,2.09,632.1
Swiggy,5,54827915,1775999.0,5762507.5,17963,1.135828535E7,3.24,3.24,1.97,632.32


In [0]:
DESCRIBE marketing_gold_campaign_performance;



col_name,data_type,comment
campaign_id,string,null
campaign_name,string,null
brand,string,null
product,string,null
total_impressions,bigint,null
total_clicks,double,null
total_spend,double,null
total_orders,bigint,null
total_sales,double,null
ctr,double,null


In [0]:
INSERT OVERWRITE TABLE marketing_gold_campaign_performance
(
    campaign_id,
    campaign_name,
    brand,
    product,
    total_impressions,
    total_clicks,
    total_spend,
    total_orders,
    total_sales,
    ctr,
    cpc,
    roas,
    aov,
    platform,
    month
)

SELECT
    campaign_id,
    campaign_name,
    brand,
    product,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(
        SUM(clicks) / NULLIF(SUM(impressions), 0) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(spend) / NULLIF(SUM(clicks), 0),
        2
    ) AS cpc,

    ROUND(
        SUM(sales) / NULLIF(SUM(spend), 0),
        2
    ) AS roas,

    ROUND(
        SUM(sales) / NULLIF(SUM(orders), 0),
        2
    ) AS aov,

    platform,

    date_format(to_date(date), 'yyyy-MM') AS month

FROM silver_ads

GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product,
    platform,
    date_format(to_date(date), 'yyyy-MM');

num_affected_rows,num_inserted_rows
300,300


In [0]:
SELECT
    month,
    platform,
    ROUND(SUM(total_spend), 2) AS total_spend,
    ROUND(SUM(total_sales), 2) AS total_sales,
    SUM(total_orders) AS total_orders,
    ROUND(SUM(total_sales) / NULLIF(SUM(total_spend), 0), 2) AS roas
FROM marketing_gold_campaign_performance
GROUP BY
    month,
    platform
ORDER BY
    month,
    total_sales DESC;

month,platform,total_spend,total_sales,total_orders,roas
2026-04,Blinkit,891900.0,2393777.81,3856,2.68
2026-04,Amazon,644150.0,1440699.62,2321,2.24
2026-04,Flipkart,515320.0,1050109.93,1690,2.04
2026-04,Zepto,426130.0,804821.6,1295,1.89
2026-04,Swiggy,346850.0,603369.92,971,1.74
2026-05,Blinkit,943825.0,2579667.41,4080,2.73
2026-05,Amazon,675580.0,1530915.35,2422,2.27
2026-05,Flipkart,546425.0,1129622.78,1786,2.07
2026-05,Zepto,447075.0,866472.01,1369,1.94
2026-05,Swiggy,367595.0,657630.05,1039,1.79


In [0]:
INSERT OVERWRITE TABLE marketing_gold_campaign_performance
(
    campaign_id,
    campaign_name,
    brand,
    product,
    total_impressions,
    total_clicks,
    total_spend,
    total_orders,
    total_sales,
    ctr,
    cpc,
    roas,
    aov,
    platform,
    month
)

SELECT
    campaign_id,
    campaign_name,
    brand,
    product,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(
        SUM(clicks) / NULLIF(SUM(impressions), 0) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(spend) / NULLIF(SUM(clicks), 0),
        2
    ) AS cpc,

    ROUND(
        SUM(sales) / NULLIF(SUM(spend), 0),
        2
    ) AS roas,

    ROUND(
        SUM(sales) / NULLIF(SUM(orders), 0),
        2
    ) AS aov,

    platform,

    DATE_FORMAT(TO_DATE(date), 'yyyy-MM') AS month

FROM silver_ads

GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product,
    platform,
    DATE_FORMAT(TO_DATE(date), 'yyyy-MM');

num_affected_rows,num_inserted_rows
300,300
